# Week 1 — SELECT, WHERE, ORDER BY, LIMIT: Reading Data
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

By the end of this session, you will be able to:
- Write SELECT with WHERE, ORDER BY, and LIMIT from memory

Today we start reading data with raw SQL against the same Olist dataset you already
know from Phase 2a — except this time you'll talk to the database directly instead
of going through pandas.

### Setup — run this cell first

This loads the 8 Olist tables into a file-based SQLite database and connects the
`%%sql` cell magic to it. Because `autopandas` is on, every `%%sql` query returns a
pandas DataFrame under the hood — useful later when we want to inspect a result in
Python — but for now we'll mostly just look at the tables SQL prints back to us.

In [ ]:
"""
Standard Olist SQL Setup for Google Colab — Phase 2b SQL

Use this block as the FIRST code cell in every Phase 2b notebook (all weeks).

Design notes (why this differs from the Phase 2a pandas setup):
- We teach SQL with the `%%sql` cell magic (jupysql) so beginners write raw
  SQL, not `pd.read_sql("...")` strings. Students meet SQL syntax first;
  the Python wrapper comes later (Phase 2c).
- jupysql opens its OWN database connection. A `sqlite3.connect(":memory:")`
  database is private to that one connection, so a separate `%sql sqlite://`
  connection would see ZERO tables. We therefore build a **file-based**
  SQLite database at /content/olist.db and point both pandas (for loading)
  and jupysql (for querying) at the same file.
- `autopandas = True` makes every `%%sql` cell return a pandas DataFrame, so
  self-check cells can `assert` on `.iloc`/`.shape` directly.

After this cell runs, query with:

    %%sql
    SELECT * FROM orders LIMIT 5

Or capture a result for checking:

    %%sql result <<
    SELECT COUNT(*) AS n FROM orders WHERE order_status = 'delivered'
    # then in a plain Python cell:  assert int(result.iloc[0]['n']) == 96478
"""

import os
import sqlite3
import pandas as pd
from google.colab import drive

# 1) Mount Drive and locate the Olist CSVs
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/cohort7/datasets/olist"

# 2) Build a file-based SQLite database (shared by pandas + jupysql)
DB_PATH = "/content/olist.db"
conn = sqlite3.connect(DB_PATH)

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")

conn.close()
print("\nDatabase ready.")

# 3) Connect jupysql to the SAME database file
%load_ext sql
%config SqlMagic.autopandas = True          # %%sql cells return DataFrames
%config SqlMagic.feedback = 0               # quieter output
%sql sqlite:////content/olist.db

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


## Why this matters

The `orders` table holds all **99,441** orders Olist has ever processed, but only
**96,478** of them were actually delivered to a customer — the rest were canceled,
lost, or are still in transit. If someone in ops asks *"how many orders never made it
to the customer?"*, scrolling through 99,441 rows in a spreadsheet is not an option.
`SELECT` and `WHERE` are how we ask the database that question directly and get back
exactly the rows — or the count — we need, in milliseconds. Today's four clauses
(`SELECT`, `WHERE`, `AND`/`OR`, `ORDER BY` + `LIMIT`) are the four moves you will use
in almost every query you ever write.

## 1. `SELECT` — choosing columns

`SELECT` is how you tell the database *which columns* you want back. `SELECT *`
means "give me every column, I haven't decided what I need yet" — it's the SQL
equivalent of opening a spreadsheet and scrolling right to see everything. Once you
know exactly which columns matter for the question you're answering, you name them
explicitly instead — this is faster for the database to return and much easier for a
human to read, especially on a table with a dozen-plus columns like `orders`. Think
of `SELECT *` as "show me the whole filing cabinet" and `SELECT col1, col2` as "just
hand me these two folders.

In [ ]:
%%sql
-- SELECT * : every column, useful for a first look at a table you don't know yet
SELECT *
FROM orders
LIMIT 10

In [ ]:
%%sql
-- Naming columns explicitly: only what we actually need for the question at hand
SELECT order_id, customer_id, order_status, order_purchase_timestamp
FROM orders
LIMIT 10

## 2. `WHERE` — filtering rows

`SELECT` picks columns; `WHERE` picks *rows*. It's a boolean test applied to every
row in the table — if the condition is true for that row, the row stays in the
result; if it's false, it's dropped. This is exactly like Excel's filter dropdown,
except instead of clicking checkboxes you write the condition as text:
`order_status = 'delivered'` reads almost like English. Below we filter to just the
delivered orders, then count how many rows survive the filter under a few different
conditions — these three counts are all independently verified against the dataset,
so if your query returns something else, the query (not the number) is wrong.

In [ ]:
%%sql
-- Filter to only delivered orders and look at a sample of the rows
SELECT order_id, customer_id, order_status
FROM orders
WHERE order_status = 'delivered'
LIMIT 10

In [ ]:
%%sql
-- How many orders are delivered, in total?
SELECT COUNT(*) AS total
FROM orders
WHERE order_status = 'delivered'   -- Expected: 96,478

In [ ]:
%%sql
-- != means "not equal to" — every order that is anything OTHER than delivered
SELECT COUNT(*) AS total
FROM orders
WHERE order_status != 'delivered'   -- Expected: 2,963

## 3. `AND` / `OR` in `WHERE`

Real business questions rarely stop at one condition. `AND` requires *every* listed
condition to be true for a row to survive — it narrows the result. `OR` requires
*at least one* of the listed conditions to be true — it widens the result to include
either group. A useful mental model: `AND` is the intersection of two filters, `OR`
is the union. Here we want every order that went wrong in one of two ways — canceled
or unavailable — so we reach for `OR`.

In [ ]:
%%sql
-- OR widens the filter: canceled orders UNION unavailable orders
SELECT COUNT(*) AS total
FROM orders
WHERE order_status = 'canceled'
   OR order_status = 'unavailable'   -- Expected: 1,234 (625 canceled + 609 unavailable)

## 4. `ORDER BY` and `LIMIT`

By default, a database returns rows in whatever order it happens to store them —
which is not useful when you want "the top 10 highest payments" or "the 5 oldest
orders." `ORDER BY column DESC` sorts rows from highest to lowest on that column
(`ASC`, ascending, is the default if you omit it); `LIMIT n` then caps the result to
just the first `n` rows after sorting. The two clauses almost always travel
together: sort first, then cut off at the number of rows you actually want to see.

In [ ]:
%%sql
-- Sort payments highest-first, keep only the top 10
SELECT order_id, payment_type, payment_value
FROM order_payments
ORDER BY payment_value DESC
LIMIT 10

## Going deeper — integer division silently truncates

SQLite (like most SQL) does *integer* division when both operands are whole numbers:
`5 / 2` gives `2`, not `2.5` — the decimal part is just thrown away. This bites
people the moment they try to compute a percentage. The fix is to force at least one
side to be a real (floating-point) number by multiplying by `1.0` before dividing.
Below, we compute delivered orders as a share of all orders using our two verified
base counts (96,478 delivered out of 99,441 total) — watch what changes when we add
`* 1.0`.

In [ ]:
%%sql
-- WRONG (commented out) — integer division truncates to 0:
--   SELECT (SELECT COUNT(*) FROM orders WHERE order_status = 'delivered')
--          / (SELECT COUNT(*) FROM orders) AS delivered_share
-- CORRECT — multiply by 1.0 to force real division:
SELECT
  (SELECT COUNT(*) FROM orders WHERE order_status = 'delivered') * 1.0
  / (SELECT COUNT(*) FROM orders) AS delivered_share

## Common mistake — `= NULL` never matches

A beginner instinct is to filter for missing values with `column = NULL`. In SQL,
`NULL` means "unknown," and an unknown is never considered equal to anything — not
even to another `NULL`. `column = NULL` silently returns **zero rows every time**,
with no error to warn you. The correct way to test for missing data is the dedicated
`IS NULL` (or `IS NOT NULL`) operator. We can see this directly on
`order_delivered_customer_date`, which is `NULL` whenever an order was never actually
delivered to the customer.

In [ ]:
%%sql
-- ── COMMON MISTAKE ──────────────────────────────────
-- WRONG — matches ZERO rows, NULL is never '= NULL':
--   SELECT COUNT(*) FROM orders WHERE order_delivered_customer_date = NULL
-- CORRECT — use IS NULL:
SELECT COUNT(*) AS total
FROM orders
WHERE order_delivered_customer_date IS NULL   -- Expected: 2,965

## Mini-challenge — your turn

⏱ ~5 minutes

Using only `SELECT` and `WHERE`, count how many payments in `order_payments` were
made using `boleto` (a common Brazilian payment method, similar to a bank slip).

**Expected:** 19,784 boleto payments.

In [ ]:
%%sql
-- Your query here


## Session Summary

| Clause | What it does | Example |
|---|---|---|
| `SELECT` | choose columns | `SELECT order_id, order_status FROM orders` |
| `WHERE` | filter rows | `WHERE order_status = 'delivered'` |
| `AND` | narrow — every condition must be true | `WHERE a = 1 AND b = 2` |
| `OR` | widen — at least one condition must be true | `WHERE order_status = 'canceled' OR order_status = 'unavailable'` |
| `ORDER BY` | sort rows (`ASC`/`DESC`) | `ORDER BY payment_value DESC` |
| `LIMIT` | cap the number of rows returned | `LIMIT 10` |
| `IS NULL` | test for missing values (never `= NULL`) | `WHERE order_delivered_customer_date IS NULL` |

---
**Coming up Thursday**: independent practice — you'll apply `SELECT`, `WHERE`,
`ORDER BY`, and `LIMIT` on your own, without a demo to follow, to answer business
questions about Olist orders, payments, and reviews (group work).

Thursday is a **group exercise**: working in small groups, you'll apply today's
clauses to answer 5 business questions — NULL delivery dates, the top-5 priced
items, boleto payments, SP-state customers, and the worst (1-star) reviews —
using only what you learned today.